<a href="https://colab.research.google.com/github/zeeshanhere10/abcde-git/blob/master/getting_started_tutorials/cudf_pandas_stocks_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv("train.csv", parse_dates=["Date"])
df = df.sort_values("Date")

train = df[df["Date"] < "2015-06-01"]
val   = df[(df["Date"] >= "2015-06-01") & (df["Date"] < "2015-07-01")]
test  = df[df["Date"] >= "2015-07-01"]

print(train.shape, val.shape, test.shape)


/tmp/ipython-input-3881429292.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train.csv", parse_dates=["Date"])


(436608, 9) (33450, 9) (34565, 9)


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Create the LEAKAGE FEATURE (this uses future data!)
df["future_sales_mean"] = df.groupby("Store")["Sales"].transform("mean")

# Now split AFTER creating the leaked feature
train = df[df["Date"] < "2015-06-01"]
val   = df[(df["Date"] >= "2015-06-01") & (df["Date"] < "2015-07-01")]
test  = df[df["Date"] >= "2015-07-01"]

# Select features for training
features = ["Store", "DayOfWeek", "Open", "Promo", "future_sales_mean"]

# Prepare data
X_train = train[features]
y_train = train["Sales"]

X_val = val[features]
y_val = val["Sales"]

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_val)

# Evaluate - you'll see AMAZING results (TOO GOOD!)
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"MAE: {mae:.2f}")
print(f"R² Score: {r2:.4f}")
print("\n🚨 These results look TOO GOOD - that's the leakage!")

MAE: 577.25
R² Score: 0.9434

🚨 These results look TOO GOOD - that's the leakage!
